In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
import librosa
import math
import traceback
import gc
from torch.utils.data import DataLoader, Dataset
#from vampnet.interface import Interface # La classe principale di VampNet
#from audiocraft.data.audio import audio_write

from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance, TimeSeriesResampler
from tslearn.shapelets import LearningShapelets

from transformers import ClapModel, ClapProcessor

import matplotlib.pyplot as plt

/home/marco/Documenti/data_mining/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-17 10:31:33.154952: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-17 10:31:33.756455: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-17 10:31:36.827710: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You

In [2]:

MP3_FOLDER = "/Users/lorenzoallegrini/Downloads/fedez_fibra"
DATASETS_FOLDER = "../datasets"

ARTIST_MAPPING = {
    "07024718": "Fedez",
    "25707984": "Fabri Fibra"
}

def get_artist_from_filename(filename):
    try:
        # Nota: La funzione lavora solo sul nome del file, non sul path intero
        artist_id = filename.split(' - ')[0].replace('ART', '')
        return ARTIST_MAPPING.get(artist_id, "Unknown")
    except:
        return "Unknown"


filenames = [f for f in os.listdir(MP3_FOLDER) if f.endswith('.mp3')]
print(f"Found {len(filenames)} MP3 files.")


fedez_files = [
    os.path.join(MP3_FOLDER, f)
    for f in filenames
    if get_artist_from_filename(f) == "Fedez"
]

fibra_files = [
    os.path.join(MP3_FOLDER, f)
    for f in filenames
    if get_artist_from_filename(f) == "Fabri Fibra"
]

all_files = fedez_files[:15] + fibra_files[:15]



FileNotFoundError: [Errno 2] No such file or directory: '/Users/lorenzoallegrini/Downloads/fedez_fibra'

In [ ]:
class ArtistSongDatasetSliding(Dataset):
    def __init__(self, audio_path: str, chunk_duration=10, stride_duration=5, target_sr=32000):
        self.chunk_duration = chunk_duration
        self.stride_duration = stride_duration
        self.target_sr = target_sr

        try:
            self.full_audio, _ = librosa.load(audio_path, sr=self.target_sr, mono=True)
            self.full_audio_t = torch.from_numpy(self.full_audio).float().unsqueeze(0).unsqueeze(0) # [1, 1, T]
        except Exception as e:
            print(f"Error loading librosa: {e}")
            self.full_audio_t = None

        self.chunk_samples = int(self.target_sr * self.chunk_duration)
        self.stride_samples = int(self.target_sr * self.stride_duration)
        
        if self.full_audio_t is not None:
            total_samples = self.full_audio_t.shape[-1]
            if total_samples <= self.chunk_samples:
                self.n_windows = 1 if total_samples > 0 else 0
            else:
                self.n_windows = math.ceil((total_samples - self.chunk_samples) / self.stride_samples) + 1
        else:
            self.n_windows = 0

    def __len__(self):
        return self.n_windows

    def __getitem__(self, index):
        start_idx = index * self.stride_samples
        end_idx = start_idx + self.chunk_samples

        audio_chunk = self.full_audio_t[..., start_idx : end_idx]

        if audio_chunk.shape[-1] == 0: return None

        if audio_chunk.shape[-1] < self.chunk_samples:
            padding = self.chunk_samples - audio_chunk.shape[-1]
            audio_chunk = torch.nn.functional.pad(audio_chunk, (0, padding))

        return {
            "signal": audio_chunk.squeeze(0), # [1, T]
            "chunk_index": index
        }

In [ ]:
def predict_masked_window(loader, interface, mask_duration_sec):
    interface.model.eval() 
    prediction_errors = []
    
    codec = interface.codec
    frame_rate = codec.frame_rate 
    mask_tokens_len = int(mask_duration_sec * frame_rate)
    
    device = interface.device
    criterion = nn.CrossEntropyLoss(reduction='none')

    with torch.inference_mode():
        for batch in loader:
            if batch is None: continue

            signal = batch['signal'].to(device)

            z = interface.to_seq(signal) 
            
            B, n_codebooks, T_tokens = z.shape
            
            chunk_losses = np.zeros(T_tokens)
            counts = np.zeros(T_tokens)

            step_size = mask_tokens_len 
            
            for t_start in range(0, T_tokens, step_size):
                t_end = min(t_start + mask_tokens_len, T_tokens)
                if t_start >= t_end: break
                mask = torch.zeros((B, T_tokens), dtype=torch.bool, device=device)
                mask[:, t_start:t_end] = True 

                logits = interface.model(z, mask=mask)
                
                logits_window = logits[:, :, t_start:t_end, :] # [B, K, window, V]
                target_window = z[:, :, t_start:t_end]         # [B, K, window]
                
                logits_flat = logits_window.reshape(-1, logits_window.shape[-1])
                target_flat = target_window.reshape(-1)
                
                loss = criterion(logits_flat, target_flat)
                
                loss_reshaped = loss.view(B, n_codebooks, -1) 
                loss_per_time = loss_reshaped.mean(dim=1).squeeze(0) #  avg on 4 codebook -> [window]
                
                chunk_losses[t_start:t_end] = loss_per_time.cpu().numpy()
            
            prediction_errors.extend(chunk_losses)
            
            del z, logits, mask, signal
            
    return prediction_errors


In [ ]:
"""model_id = "hugggof/vampnet-base-best" 
chunk_sec = 10     
stride_sec = 5       

interface = Interface.from_pretrained(model_id)
interface.to("cuda" if torch.cuda.is_available() else "cpu")

dataset_path = "./tuo_dataset_audio"


for filename in os.listdir(dataset_path):
    if not filename.endswith((".mp3", ".wav", ".flac")): continue
    
    file_path = os.path.join(dataset_path, filename)
    print(f"\nProcessing: {filename}")
    
    try:
        dataset = ArtistSongDatasetSliding(
            file_path, 
            chunk_duration=chunk_sec, 
            stride_duration=stride_sec
        )
        
        if len(dataset) == 0: continue
        
        loader = DataLoader(dataset, batch_size=1, shuffle=False)
        
        # Analisi
        raw_errors = predict_masked_window(loader, interface, mask_window_sec)
        
        # Analisi Statistica
        full_errors = np.array(raw_errors)
        
        mean_val = np.mean(full_errors)
        std_val = np.std(full_errors)
        thresh = mean_val + (2.5 * std_val) # Soglia per outlier
        
        anomalies = np.where(full_errors > thresh)[0]
        
        print(f" -> Mean Loss: {mean_val:.3f} (Più è alta, più è 'strano' lo stile)")
        print(f" -> Peak Anomaly Score: {np.max(full_errors):.3f}")
        
        # Convertiamo indici in secondi (VampNet frame rate ~50Hz)
        if len(anomalies) > 0:
            frame_rate = interface.codec.frame_rate
            sec = anomalies[0] / frame_rate
            print(f" -> Primo momento anomalo a: {sec:.2f}s")
            
    except Exception as e:
        print(f"ERRORE: {e}")
        traceback.print_exc()
    finally:
        torch.cuda.empty_cache()
        gc.collect()"""

## Anomaly detection with matrix profile

In [ ]:
from transformers import ClapModel, ClapProcessor
model = ClapModel.from_pretrained("laion/clap-htsat-unfused")
processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")

In [ ]:

def extract_window_clap(model, processor, windowed_audio, prompt="chorus", sr=48000):
    inputs = processor(
        text=[prompt],
        audio=windowed_audio,
        return_tensors="pt",
        padding=True,
        sampling_rate=sr
    )
    outputs = model(**inputs)
    probs = outputs.logits_per_text.softmax(dim=1)
    return probs[0]


def scan_audio_for_prompt(model, processor, y, sr=48000, prompt="chorus", extract_func=extract_window_clap, window_sec=10, stride_sec=2):
    window_samples = int(window_sec * sr)
    stride_samples = int(stride_sec * sr)

    audio_chunks = []
    time_points = []
    
    for i in range(0, len(y) - window_samples, stride_samples):
        chunk = y[i : i + window_samples]
        audio_chunks.append(chunk)
        time_points.append(i / sr) 
    
    probs = extract_func(prompt, audio_chunks, sr)

    scores = probs.detach().cpu().numpy()
    best_idx = np.argmax(scores)
    best_time = time_points[best_idx]
    best_score = scores[best_idx]
    
    return best_time, best_score, time_points, scores

def plot_search_results(time_points, scores, prompt, best_time, window_sec):
    plt.figure(figsize=(10, 4))
    plt.plot(time_points, scores, label='CLAP Confidence', color='purple')
    
    plt.axvline(best_time, color='r', linestyle='--', label=f'Best Match: {best_time:.1f}s')
    plt.axvspan(best_time, best_time + window_sec, color='red', alpha=0.2, label='Window')
    
    plt.title(f"Temporal research: '{prompt}'")
    plt.xlabel("Time")
    plt.ylabel("Probability")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
target_prompt = "energetic chorus of the song"

for filepath in all_files:
    filename = os.path.basename(filepath)
    print(f"\nProcessing: {filename}")
    
    try:
        y, sr = librosa.load(filepath, sr=48000)
        
        best_time, best_score, times, scores = scan_audio_for_prompt(
            model=model,         
            processor=processor,  
            y=y, 
            sr=sr, 
            prompt=target_prompt, 
            window_sec=10, 
            stride_sec=2
        )
        
        print(f"--> Chorus trovato a {best_time:.1f}s (Confidenza: {best_score:.2%})")
        
    except Exception as e:
        print(f"Errore su {filename}: {e}")

In [ ]:
def extract_envelope_features(target_files, original_sr=24000, target_sr=10):
    hop_length = int(sr / target_sr)
    features = []
    for filepath in target_files:
        y, sr = librosa.load(filepath, sr=original_sr)
        env = librosa.feature.rms(y=y, frame_length=hop_length*2, hop_length=hop_length)
        features.append(env.flatten())
            
    return np.array(features)

In [ ]:
import numpy as np

def get_prototypes_dtw(X, kmeans_model, predicted_clusters, samples_per_cluster=15):
    cluster_samples = []     
    real_centers_indices = [] 
    
    n_clusters = kmeans_model.n_clusters
    
    distances_matrix = kmeans_model.transform(X)

    for c in range(n_clusters):
        idx_in_cluster = np.where(predicted_clusters == c)[0]
        
        if len(idx_in_cluster) == 0:
            cluster_samples.append([])
            real_centers_indices.append(None)
            continue

        dists_to_center = distances_matrix[idx_in_cluster, c]
        
        closest_idx_local = np.argsort(dists_to_center)

        best_match_local_idx = closest_idx_local[0]
        best_match_global_idx = idx_in_cluster[best_match_local_idx]
        real_centers_indices.append(best_match_global_idx)
        
        top_n_local = closest_idx_local[:samples_per_cluster]
        top_n_global = idx_in_cluster[top_n_local]
        cluster_samples.append(top_n_global)
    

    return cluster_samples, real_centers_indices

In [ ]:

def plot_clusters_scatter(reducer, embeddings, predicted_clusters, figsize=(10,6), samples_per_cluster=15):
    X_umap = reducer.fit_transform(embeddings)

    unique_clusters = np.unique(predicted_clusters)

    cluster_samples = []     
    real_centers_indices = [] 

    for c in unique_clusters:
        idx_cluster = np.where(predicted_clusters == c)[0]
        
        X_c = embeddings[idx_cluster]
        centroid = X_c.mean(axis=0)
        
        dist_to_centroid = np.linalg.norm(X_c - centroid, axis=1)

        closest_idx_local = np.argsort(dist_to_centroid)

        best_local = closest_idx_local[0]
        best_global = idx_cluster[best_local]
        real_centers_indices.append(best_global)

        top_n_local = closest_idx_local[:samples_per_cluster]
        top_n_global = idx_in_cluster = idx_cluster[top_n_local]
        cluster_samples.append(top_n_global)


    plt.figure(figsize=figsize)
    sns.scatterplot(
        x=X_umap[:, 0], 
        y=X_umap[:, 1], 
        hue=predicted_clusters, 
        palette='colorblind',
        s=100,
        legend='full',
        alpha=0.6
    )

    plt.title('Cluster Projection (Fedez vs Fibra)')
    plt.xlabel("Dim 1")
    plt.ylabel("Dim 2")
    plt.legend()
    plt.show()

    return cluster_samples, real_centers_indices

In [ ]:

def plot_centroids_envelopes(envelopes, cluster_ids, sr=10, title="Cluster centroid envelope comparison"):
    """
    Plotta solo le linee dei centroidi per un confronto pulito.
    """
    plt.figure(figsize=(12, 6))
    
    # Asse X in secondi
    time_axis = np.arange(envelopes.shape[1]) / sr
    
    # Colori distinti
    colors = plt.cm.tab10.colors
    
    # Loop su ogni centroide estratto
    for i, cluster_id in enumerate(cluster_ids):
        # Rimuovi dimensioni inutili
        y_data = envelopes[i].flatten()
        
        # Plot della linea
        plt.plot(time_axis, y_data, 
                 color=colors[i % 10], 
                 linewidth=3, 
                 alpha=0.9, 
                 label=f"Cluster {cluster_id} (Centroide)")

    plt.title(title, fontsize=16)
    plt.xlabel("Tempo (Secondi)")
    plt.ylabel("Volume (RMS)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:

reducer = umap.UMAP(
    n_neighbors=30,  
    min_dist=0.1, 
    n_components=2, 
    random_state=42
)

cluster_samples, centroid_indices = plot_clusters_scatter(reducer, mert_wav2vec2_embeddings_pca, predicted_clusters)

centroid_paths = [all_files[i] for i in centroid_indices]

centroids_audio_envelope = extract_envelope_features(centroid_paths, target_sr=10)

unique_clusters = np.sort(np.unique(predicted_clusters))

plot_centroids_comparison(
    envelopes=centroids_audio_envelope, 
    cluster_ids=unique_clusters, 
    sr=10, 
    title="Confronto Prototipi: Fedez vs Fibra"
)